# Local/Global World-Model Planning Foundation

Operational notebook for the local/global track: the DINO-WM latent world model (trained by notebook 02) is the trusted **global** forward model, a small differentiable surrogate trained on cached DINO latents is the **local** model, and hybrid planners combine global CEM search with local gradient refinement plus global re-scoring. Training, transition export, and planning evaluation are delegated to `scripts/local_global/`; heavy launches are gated behind `RUN_LOCAL_GLOBAL=1`. See `LOCAL_GLOBAL_DINO_WM_IMPLEMENTATION_SPEC.md` for the design brief.

In [2]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/Thomas-Georges/World_Models_LAS.git"
REPO_DIR = Path("/content/World_Models_LAS")
REPO_BRANCH = "main"
REPO_RUN_CWD = REPO_DIR.parent


def ensure_run_cwd() -> None:
    REPO_RUN_CWD.mkdir(parents=True, exist_ok=True)
    os.chdir(REPO_RUN_CWD)


def run_git(args, *, check=True):
    ensure_run_cwd()
    print("$", " ".join(map(str, args)))
    result = subprocess.run(
        args,
        check=False,
        cwd=str(REPO_RUN_CWD),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, args, output=result.stdout)
    return result


def clone_repo():
    ensure_run_cwd()
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_git(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])


def sync_repo():
    ensure_run_cwd()
    if not (REPO_DIR / ".git").is_dir():
        clone_repo()
        return

    run_git(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])
    fetch = run_git(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=False)
    if fetch.returncode != 0:
        print(f"Fetch failed for existing checkout at {REPO_DIR}; replacing it with a fresh clone.")
        clone_repo()
        return

    run_git(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_git(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"])


sync_repo()


command = ["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"]
print("$", " ".join(command))
head = subprocess.check_output(command, text=True, cwd=str(REPO_RUN_CWD)).strip()
print("Repository HEAD:", head)

$ git clone --branch main --single-branch https://github.com/Thomas-Georges/World_Models_LAS.git /content/World_Models_LAS
Cloning into '/content/World_Models_LAS'...
$ git -C /content/World_Models_LAS log --oneline -1
Repository HEAD: dc53d2e Adding a safeguard for torch-torchvision mismatch


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
import os

DRIVE_MOUNT = Path("/content/drive")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()

if IN_COLAB:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT), force_remount=False)
    if not (DRIVE_MOUNT / "MyDrive").exists():
        raise RuntimeError("Google Drive mount did not expose /content/drive/MyDrive.")
    print(f"Google Drive mounted at {DRIVE_MOUNT}")
else:
    print("Not running in Colab; skipping Google Drive mount.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at /content/drive


In [5]:
from pathlib import Path
import os
import sys

REPO = REPO_DIR if 'REPO_DIR' in globals() and (REPO_DIR / '.git').is_dir() else Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

DRIVE_ROOT = Path(os.environ.get('WM_POC_DRIVE_ROOT', '/content/drive/MyDrive/wm_poc'))
os.environ.setdefault('DINO_WM_REPO', str(DRIVE_ROOT / 'external_repos/dino_wm'))
os.environ.setdefault('DINO_WM_DATA_ROOT', str(DRIVE_ROOT / 'data/dino_wm'))
os.environ.setdefault('DINO_CKPT_ROOT', str(DRIVE_ROOT / 'checkpoints/dino_wm'))
os.environ.setdefault('DINO_WM_FEATURE_CACHE', '/content/wm_poc_latent_cache')
os.environ.setdefault('LG_RUN_ROOT', str(DRIVE_ROOT / 'logs/local_global'))

CONFIG_SMOKE_SYNTHETIC = REPO / 'configs/local_global/smoke_synthetic.yaml'
CONFIG_SMOKE_POINTMAZE = REPO / 'configs/local_global/smoke_pointmaze.yaml'
CONFIG_POINTMAZE_T4 = REPO / 'configs/local_global/pointmaze_surrogate_t4.yaml'
CONFIG_POINTMAZE_A100 = REPO / 'configs/local_global/pointmaze_surrogate_a100.yaml'
LG_RUN_ROOT = Path(os.environ['LG_RUN_ROOT'])

print(f"REPO:        {REPO}")
print(f"DRIVE_ROOT:  {DRIVE_ROOT}")
print(f"LG_RUN_ROOT: {LG_RUN_ROOT}")
for name in ('DINO_WM_REPO', 'DINO_WM_DATA_ROOT', 'DINO_CKPT_ROOT', 'DINO_WM_FEATURE_CACHE'):
    print(f"{name}={os.environ[name]}")

REPO:        /content/World_Models_LAS
DRIVE_ROOT:  /content/drive/MyDrive/wm_poc
LG_RUN_ROOT: /content/drive/MyDrive/wm_poc/logs/local_global
DINO_WM_REPO=/content/drive/MyDrive/wm_poc/external_repos/dino_wm
DINO_WM_DATA_ROOT=/content/drive/MyDrive/wm_poc/data/dino_wm
DINO_CKPT_ROOT=/content/drive/MyDrive/wm_poc/checkpoints/dino_wm
DINO_WM_FEATURE_CACHE=/content/wm_poc_latent_cache


In [6]:
import shlex
import subprocess


def show_command(cmd, *, env=None):
    prefix = ' '.join(f"{key}={shlex.quote(str(value))}" for key, value in (env or {}).items())
    rendered = shlex.join(str(part) for part in cmd)
    print(f"$ {prefix + ' ' if prefix else ''}{rendered}")


def run_if_safe(cmd, *, env=None, check=False):
    # Stream the child's output into the cell: Colab does not render output a
    # subprocess writes to the kernel's inherited stdout fd, so a plain
    # subprocess.run looks like the command did nothing.
    show_command(cmd, env=env)
    # Always run from the repo root: a relative script path must resolve even
    # if the os.chdir(REPO) setup cell was not run in this session.
    cwd = str(REPO) if 'REPO' in globals() else None
    process = subprocess.Popen(
        [str(part) for part in cmd],
        env={**os.environ, **(env or {})},
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
    process.wait()
    print(f"[exit code {process.returncode}]")
    if check and process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, [str(part) for part in cmd])
    return process


def enable_local_global_runs():
    os.environ['RUN_LOCAL_GLOBAL'] = '1'
    print('RUN_LOCAL_GLOBAL=1: heavy local/global launches are armed for this session.')


def run_lg(cmd, *, env=None):
    if os.environ.get('RUN_LOCAL_GLOBAL') != '1':
        show_command(cmd, env=env)
        print('Local/global execution is disabled for safety.')
        print('Run enable_local_global_runs() and rerun this cell to launch.')
        return None
    return run_if_safe(cmd, env=env, check=True)

## Environment and Drive Setup

In [7]:
run_if_safe([sys.executable, str(REPO / 'scripts/verify_environment.py'), '--cpu-only'])
run_if_safe([sys.executable, str(REPO / 'scripts/verify_drive_layout.py'), '--dry-run'])

$ /usr/bin/python3 /content/World_Models_LAS/scripts/verify_environment.py --cpu-only
Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Working directory: /content/World_Models_LAS
PyTorch: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
GPU: NVIDIA A100-SXM4-40GB
nvidia-smi output:
Wed Jun 17 19:31:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+====

<Popen: returncode: 0 args: ['/usr/bin/python3', '/content/World_Models_LAS/...>

## Config Selection and Display

In [8]:
# Both profiles run the SAME experiment (model sizes, planner budgets,
# episodes, run dir); they differ only in throughput knobs (global rollout
# chunking, dataloader workers). The GPU check below just picks the matching
# throughput profile - a T4 runs the full A100-grade experiment, only slower.
# Override with LG_CONFIG=/path/to/config.yaml before running this cell.
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else 'cpu'
    BF16_OK = HAS_CUDA and torch.cuda.get_device_capability()[0] >= 8
except ImportError:
    HAS_CUDA, GPU_NAME, BF16_OK = False, 'cpu (torch not installed)', False

CONFIG = Path(os.environ.get('LG_CONFIG', str(CONFIG_POINTMAZE_A100 if BF16_OK else CONFIG_POINTMAZE_T4)))

from wm_poc.local_global.configs import action_data_dir, latent_cache_dir, load_local_global_config


def loadable_checkpoints(run_dir: Path) -> list[Path]:
    """model_latest.pth / model_<N>.pth; the rolling model_latest_step.pth is not loadable."""
    ckpts = run_dir / 'checkpoints'
    if not ckpts.is_dir():
        return []
    return sorted(p for p in ckpts.glob('model_*.pth') if p.name != 'model_latest_step.pth')


def usable_global_runs(outputs_root: Path) -> list[str]:
    """DINO-WM runs that can drive the global model (hydra.yaml + a loadable checkpoint)."""
    runs = []
    if outputs_root.is_dir():
        for cand in sorted(outputs_root.iterdir()):
            if (cand / 'hydra.yaml').is_file() and loadable_checkpoints(cand):
                runs.append(cand.name)
    return runs


lg_config = load_local_global_config(CONFIG)
OUTPUTS_ROOT = Path(os.environ['DINO_CKPT_ROOT']) / 'outputs'
USABLE_GLOBAL_RUNS = usable_global_runs(OUTPUTS_ROOT)
configured_global = str(lg_config['global_model']['model_name'])
if (
    'LG_GLOBAL_MODEL_NAME' not in os.environ
    and lg_config['global_model']['source'] == 'dino_wm'
    and configured_global not in USABLE_GLOBAL_RUNS
    and USABLE_GLOBAL_RUNS
):
    # The default run name has no usable checkpoint, but another completed
    # DINO-WM run does (e.g. the full run was trained on a T4). Prefer
    # full-data runs over low-data scratch/fine-tune checkpoints - a fresh
    # fine-tune from notebook 02 must not silently become the global model -
    # then break ties by newest checkpoint. LG_GLOBAL_MODEL_NAME overrides.
    # Only ever auto-select a MAIN (full-data) DINO-WM run. A low-data
    # scratch/fine-tune checkpoint must never silently become the global model;
    # if no full run is usable, leave the (unusable) default so the launch is
    # skipped with instructions rather than run against the wrong model.
    preferred = [name for name in USABLE_GLOBAL_RUNS if 'full_nodecoder' in name]
    if preferred:
        newest = max(
            preferred,
            key=lambda name: max(p.stat().st_mtime for p in loadable_checkpoints(OUTPUTS_ROOT / name)),
        )
        os.environ['LG_GLOBAL_MODEL_NAME'] = newest  # propagates into the launched scripts
        lg_config = load_local_global_config(CONFIG)
        print(f"NOTE: configured global run '{configured_global}' has no usable checkpoint;")
        print(f"      auto-selected full-data run '{newest}'.")
        print('      Set LG_GLOBAL_MODEL_NAME explicitly to override this choice.')
    else:
        print(f"WARNING: no full-data (full_nodecoder) run has a usable checkpoint under {OUTPUTS_ROOT};")
        print(f"         refusing to auto-pick a low-data run. Usable runs: {USABLE_GLOBAL_RUNS}.")
        print('         Finish the main DINO-WM training in notebook 02, or set')
        print('         LG_GLOBAL_MODEL_NAME explicitly to force a specific run.')

# model_epoch=latest needs model_latest.pth; fall back to the highest numbered
# epoch checkpoint when only those exist (e.g. after a backup restore).
active_run = OUTPUTS_ROOT / str(lg_config['global_model']['model_name'])
if (
    'LG_GLOBAL_MODEL_EPOCH' not in os.environ
    and lg_config['global_model']['source'] == 'dino_wm'
    and str(lg_config['global_model']['model_epoch']) == 'latest'
    and loadable_checkpoints(active_run)
    and not (active_run / 'checkpoints' / 'model_latest.pth').is_file()
):
    epochs = [
        int(p.stem.split('_')[1])
        for p in loadable_checkpoints(active_run)
        if p.stem.split('_')[1].isdigit()
    ]
    if epochs:
        os.environ['LG_GLOBAL_MODEL_EPOCH'] = str(max(epochs))
        lg_config = load_local_global_config(CONFIG)
        print(f"NOTE: model_latest.pth missing; using model_epoch={max(epochs)}.")

RUN_NAME = lg_config['run_name']
RUN_DIR = LG_RUN_ROOT / RUN_NAME
local_cfg, global_cfg, plan_cfg = lg_config['local_model'], lg_config['global_model'], lg_config['planning']
print(f"GPU: {GPU_NAME} | throughput profile: {CONFIG.name} | run: {RUN_NAME}")
print(f"Run dir:      {RUN_DIR}")
print(f"Latent cache: {latent_cache_dir(lg_config)}")
print(f"Action data:  {action_data_dir(lg_config)}")
print(f"Global model: {global_cfg['model_name']} (epoch {global_cfg['model_epoch']}, frameskip {global_cfg['frameskip']}, "
      f"rollout chunk {global_cfg['rollout_batch_size']})")
print(f"Local model:  {local_cfg['model_type']} / {local_cfg['projection']} local_dim={local_cfg['local_dim']} "
      f"context={local_cfg['context_len']} rollout={local_cfg['rollout_steps']}")
print(f"Planning:     H={plan_cfg['horizon']} goal_steps={plan_cfg['goal_steps']} "
      f"CEM {plan_cfg['cem_population']}x{plan_cfg['cem_iters']} GD {plan_cfg['gd_iters']}@{plan_cfg['gd_lr']}")
print(f"Planners:     {', '.join(lg_config['evaluation']['planners'])}")

NOTE: configured global run 'pointmaze_full_nodecoder_bf16_a100_b32_seed0' has no usable checkpoint;
      auto-selected full-data run 'pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0'.
      Set LG_GLOBAL_MODEL_NAME explicitly to override this choice.
GPU: NVIDIA A100-SXM4-40GB | throughput profile: pointmaze_surrogate_a100.yaml | run: pointmaze_local_full_seed0
Run dir:      /content/drive/MyDrive/wm_poc/logs/local_global/pointmaze_local_full_seed0
Latent cache: /content/wm_poc_latent_cache/point_maze/dinov2_vits14_img224
Action data:  /content/drive/MyDrive/wm_poc/data/dino_wm/point_maze
Global model: pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0 (epoch latest, frameskip 5, rollout chunk 128)
Local model:  residual_mlp / grid_pool_linear local_dim=256 context=2 rollout=3
Planning:     H=5 goal_steps=5 CEM 300x10 GD 100@0.05
Planners:     global_cem, local_cem, local_gd, local_adam, hybrid_cem_local_refine, hybrid_cem_local_refine_global_rescore


## DINO-WM Dependency Verification

The global model is a DINO-WM checkpoint produced by notebook 02. This cell only checks that the upstream checkout and the checkpoint reference exist; nothing is downloaded.

In [9]:
dino_repo = Path(os.environ['DINO_WM_REPO'])
print(f"Upstream dino_wm: {dino_repo} (plan.py present: {(dino_repo / 'plan.py').is_file()})")

ckpt_root = Path(os.environ['DINO_CKPT_ROOT'])
outputs_root = ckpt_root / 'outputs'
run_output = outputs_root / str(global_cfg['model_name'])
hydra_cfg = run_output / 'hydra.yaml'
epoch_ckpts = loadable_checkpoints(run_output)
print(f"Global run output: {run_output}")
print(f"  hydra.yaml: {hydra_cfg.is_file()} | loadable checkpoints: {len(epoch_ckpts)}")
if not hydra_cfg.is_file() or not epoch_ckpts:
    print('  -> This run cannot drive the global model yet (needs hydra.yaml AND model_*.pth).')
    if outputs_root.is_dir():
        print(f"  Runs under {outputs_root} (name | hydra.yaml | loadable ckpts | in backups):")
        for cand in sorted(outputs_root.iterdir()):
            if not cand.is_dir():
                continue
            n_ckpts = len(loadable_checkpoints(cand))
            # A forced fresh start moves checkpoints/ aside instead of deleting it.
            n_backup = sum(
                len(list(b.glob('model_*.pth')))
                for b in cand.glob('checkpoints_fresh_start_backup_*')
            )
            usable = ' <- usable' if (cand / 'hydra.yaml').is_file() and n_ckpts else ''
            print(f"    {cand.name} | {(cand / 'hydra.yaml').is_file()} | {n_ckpts} | {n_backup}{usable}")
        backups = sorted(run_output.glob('checkpoints_fresh_start_backup_*'))
        recoverable = [b for b in backups if any(b.glob('model_*.pth'))]
        if recoverable:
            newest_backup = recoverable[-1]
            print('  This run has checkpoints that were moved aside by a forced fresh start.')
            print('  Restore the newest backup with:')
            print(f"    mkdir -p '{run_output / 'checkpoints'}'")
            print(f"    mv '{newest_backup}'/model_*.pth '{run_output / 'checkpoints'}/'")
        print('  The config cell auto-selects the newest usable run when the default has')
        print('  no checkpoint; set LG_GLOBAL_MODEL_NAME to pick one explicitly. Only the')
        print('  planning evaluation needs the checkpoint - surrogate training does not.')
    else:
        print(f"  No runs found under {outputs_root} - train DINO-WM in notebook 02 first.")

# The in-process global model drives encode_obs on cached latents, which needs
# the marker-guarded latent bypass patch in the upstream checkout.
if (dino_repo / 'plan.py').is_file():
    run_if_safe([sys.executable, str(REPO / 'scripts/dino_wm/patch_latent_cache.py'), '--verify-only'])

Upstream dino_wm: /content/drive/MyDrive/wm_poc/external_repos/dino_wm (plan.py present: True)
Global run output: /content/drive/MyDrive/wm_poc/checkpoints/dino_wm/outputs/pointmaze_full_nodecoder_t4_fp16_b32_stride2_seed0
  hydra.yaml: True | loadable checkpoints: 11
$ /usr/bin/python3 /content/World_Models_LAS/scripts/dino_wm/patch_latent_cache.py --verify-only
DINO-WM latent cache support verified: /content/drive/MyDrive/wm_poc/external_repos/dino_wm
[exit code 0]


## Latent Cache (verify, rebuild when missing)

The local surrogate trains on the same DINOv2 latent cache as the DINO-WM track. The cache deliberately lives on session-local disk (`/content/wm_poc_latent_cache` - Drive FUSE random reads are too slow for training), so **every fresh Colab session must rebuild it once** (~10-20 min: it encodes the PointMaze episodes through frozen DINOv2). The cell below is **live and self-gating**: it skips when the cache already covers the dataset, and otherwise installs the upstream Colab dependencies and rebuilds the cache via the idempotent DINO-WM precompute script.

In [10]:
# Verify the cache, then rebuild it in this session when missing (Colab only).
from wm_poc.local_global.datasets import LATENT_MANIFEST_NAME

CONFIG_DINO_POINTMAZE = REPO / 'configs/dino_wm/pointmaze_full_nodecoder_bf16.yaml'
cache_manifest = latent_cache_dir(lg_config) / LATENT_MANIFEST_NAME
run_if_safe([sys.executable, str(REPO / 'scripts/local_global/export_transitions.py'),
             '--config', CONFIG, '--dry-run'])

if lg_config['task'] != 'point_maze':
    print('Non-PointMaze task: the scripts generate synthetic latents on demand.')
elif cache_manifest.is_file():
    print(f"Latent cache present: {cache_manifest}")
elif not Path('/content').exists():
    print('Latent cache missing and not on Colab; skipping the automatic rebuild.')
elif not (Path(os.environ['DINO_WM_REPO']) / 'plan.py').is_file():
    print('Upstream dino_wm checkout missing; run the setup cells in notebook 02 first.')
else:
    print('Latent cache missing in this fresh session; rebuilding it now (~10-20 min).')
    run_if_safe([sys.executable, str(REPO / 'scripts/dino_wm/install_colab_deps.py'), '--quiet'], check=True)
    run_if_safe([sys.executable, str(REPO / 'scripts/dino_wm/precompute_latents.py'),
                 '--config', CONFIG_DINO_POINTMAZE, '--no-dry-run'],
                env={'RUN_DINO_WM': '1'}, check=True)
    print(f"Cache rebuilt: {cache_manifest.is_file()}")

$ /usr/bin/python3 /content/World_Models_LAS/scripts/local_global/export_transitions.py --config /content/World_Models_LAS/configs/local_global/pointmaze_surrogate_a100.yaml --dry-run
Task:          point_maze
Latent cache:  /content/wm_poc_latent_cache/point_maze/dinov2_vits14_img224
Action data:   /content/drive/MyDrive/wm_poc/data/dino_wm/point_maze
Cache manifest present: False (/content/wm_poc_latent_cache/point_maze/dinov2_vits14_img224/wm_poc_latent_manifest.json)
To create it: python scripts/dino_wm/precompute_latents.py --config configs/dino_wm/pointmaze_full_nodecoder_bf16.yaml --no-dry-run
Dry run: no files written.
[exit code 0]
Latent cache missing in this fresh session; rebuilding it now (~10-20 min).
$ /usr/bin/python3 /content/World_Models_LAS/scripts/dino_wm/install_colab_deps.py --quiet
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspe

## Transition Export

In [11]:
# Builds the episode split + window index manifest under <run>/transition_data.
# Read-only over the cache; cheap enough to run unguarded once the cache exists.
run_if_safe([sys.executable, str(REPO / 'scripts/local_global/export_transitions.py'),
             '--config', CONFIG, '--run-dir', RUN_DIR])

$ /usr/bin/python3 /content/World_Models_LAS/scripts/local_global/export_transitions.py --config /content/World_Models_LAS/configs/local_global/pointmaze_surrogate_a100.yaml --run-dir /content/drive/MyDrive/wm_poc/logs/local_global/pointmaze_local_full_seed0
Task:          point_maze
Latent cache:  /content/wm_poc_latent_cache/point_maze/dinov2_vits14_img224
Action data:   /content/drive/MyDrive/wm_poc/data/dino_wm/point_maze
Exported manifest: /content/drive/MyDrive/wm_poc/logs/local_global/pointmaze_local_full_seed0/transition_data/manifest.json (144000 train / 16000 val windows, 2000 episodes)
[exit code 0]


<Popen: returncode: 0 args: ['/usr/bin/python3', '/content/World_Models_LAS/...>

## Local Model Smoke Test

In [12]:
# One random batch through the surrogate: shape checks, finite loss, nonzero
# gradient with respect to actions. No data or GPU required.
try:
    import torch

    from wm_poc.local_global.losses import combined_local_loss
    from wm_poc.local_global.models import build_local_model

    step_action_dim = int(plan_cfg['action_dim']) * int(global_cfg['frameskip'])
    model = build_local_model(
        patches=int(global_cfg['latent_patches']),
        embed_dim=int(global_cfg['latent_dim']),
        step_action_dim=step_action_dim,
        model_type=local_cfg['model_type'],
        projection=local_cfg['projection'],
        projection_grid=int(local_cfg['projection_grid']),
        local_dim=int(local_cfg['local_dim']),
        hidden_dim=int(local_cfg['hidden_dim']),
        num_layers=int(local_cfg['num_layers']),
    )
    batch = {
        'z_context': torch.randn(4, int(local_cfg['context_len']), int(global_cfg['latent_patches']), int(global_cfg['latent_dim'])),
        'z_targets': torch.randn(4, int(local_cfg['rollout_steps']), int(global_cfg['latent_patches']), int(global_cfg['latent_dim'])),
        'actions_context': torch.randn(4, int(local_cfg['context_len']) - 1, step_action_dim),
        'actions': torch.randn(4, int(local_cfg['rollout_steps']), step_action_dim, requires_grad=True),
    }
    loss, metrics = combined_local_loss(batch, model, {})
    loss.backward()
    action_grad = float(batch['actions'].grad.abs().sum())
    assert torch.isfinite(loss) and action_grad > 0
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Surrogate OK: loss={float(loss):.4f} | action-grad L1={action_grad:.3e} | {trainable:,} trainable params")
except ImportError as exc:
    print(f"Skipped (torch unavailable): {exc}")

Surrogate OK: loss=0.2755 | action-grad L1=2.909e-03 | 793,876 trainable params


## Local Surrogate Training

Smoke first (synthetic task, CPU-friendly, ~1 minute), then the gated full run. The full cell is **live and self-gating** with three branches: training *complete* (latest checkpoint at `max_steps`) is skipped, a *partial* run **resumes** from `local_latest.pt` (model + optimizer state + step counter; rolling saves every `save_every`=1000 steps, best-by-validation every `val_every`=500), and a fresh run launches otherwise. Each session is capped by `training.max_wall_minutes`, so on a slow GPU the run simply continues across sessions - an interruption costs at most `save_every` steps of progress. The training log prints steps/s and an ETA at every validation. Set `LG_FORCE_RETRAIN=1` to restart from scratch.

In [13]:
run_if_safe([sys.executable, str(REPO / 'scripts/local_global/train_local_surrogate.py'),
             '--config', CONFIG_SMOKE_SYNTHETIC, '--run-dir', LG_RUN_ROOT / 'smoke_synthetic', '--smoke'])

$ /usr/bin/python3 /content/World_Models_LAS/scripts/local_global/train_local_surrogate.py --config /content/World_Models_LAS/configs/local_global/smoke_synthetic.yaml --run-dir /content/drive/MyDrive/wm_poc/logs/local_global/smoke_synthetic --smoke
Config: /content/World_Models_LAS/configs/local_global/smoke_synthetic.yaml | task=synthetic | max_steps=200
Resumed /content/drive/MyDrive/wm_poc/logs/local_global/smoke_synthetic/checkpoints/local_latest.pt at step 200 (best val loss 0.00191).
Training already complete (200/200 steps); nothing to do.
Pass --no-resume (or use a fresh --run-dir) to retrain from scratch.
[exit code 0]


<Popen: returncode: 0 args: ['/usr/bin/python3', '/content/World_Models_LAS/...>

In [14]:
# Full surrogate training - self-gating: complete -> skip, partial -> resume
# from local_latest.pt, cache missing -> instructions, otherwise fresh launch.
max_steps = int(lg_config['training']['max_steps'])
latest_ckpt = RUN_DIR / 'checkpoints' / 'local_latest.pt'
done_step = 0
if latest_ckpt.is_file():
    import torch
    done_step = int(torch.load(latest_ckpt, map_location='cpu').get('step', 0))
cache_manifest = latent_cache_dir(lg_config) / 'wm_poc_latent_manifest.json'
force_retrain = os.environ.get('LG_FORCE_RETRAIN') == '1'

if done_step >= max_steps and not force_retrain:
    print(f"Local surrogate training complete ({done_step}/{max_steps} steps): {latest_ckpt}")
    print('Set LG_FORCE_RETRAIN=1 (or use a fresh run dir) to retrain from scratch.')
elif not cache_manifest.is_file():
    print(f"Latent cache not ready ({cache_manifest} missing); skipping the launch.")
    print('Run the latent-cache cell above first - it rebuilds the cache in this session.')
else:
    if force_retrain:
        print('LG_FORCE_RETRAIN=1: restarting training from scratch (--no-resume).')
    elif done_step:
        print(f"Partial run found: resuming from step {done_step}/{max_steps}.")
    train_cmd = [sys.executable, str(REPO / 'scripts/local_global/train_local_surrogate.py'),
                 '--config', CONFIG, '--run-dir', RUN_DIR]
    if force_retrain:
        train_cmd.append('--no-resume')
    run_if_safe([*train_cmd, '--dry-run'])
    enable_local_global_runs()
    run_lg(train_cmd)

Local surrogate training complete (20000/20000 steps): /content/drive/MyDrive/wm_poc/logs/local_global/pointmaze_local_full_seed0/checkpoints/local_latest.pt
Set LG_FORCE_RETRAIN=1 (or use a fresh run dir) to retrain from scratch.


## Validation Rollouts

In [15]:
import json

val_file = RUN_DIR / 'metrics' / 'val_rollouts.jsonl'
if val_file.is_file():
    rows = [json.loads(line) for line in val_file.read_text().splitlines() if line.strip()]
    last = rows[-1]
    print(f"Validation @ step {last['step']} ({len(rows)} evaluations logged)")
    print(f"  one-step MSE (local space): {last.get('one_step_mse'):.6f}")
    for horizon, value in enumerate(last.get('rollout_mse_per_step', []), start=1):
        print(f"  rollout step {horizon}: MSE {value:.6f}")
    print('Local-vs-global disagreement is logged during planning evaluation')
    print('(hybrid re-score stats in planning/<planner>/summary.json).')
else:
    print(f"No validation rollouts yet at {val_file}; train the surrogate first.")

Validation @ step 20000 (41 evaluations logged)
  one-step MSE (local space): 0.002615
  rollout step 1: MSE 0.002615
  rollout step 2: MSE 0.003028
  rollout step 3: MSE 0.003130
Local-vs-global disagreement is logged during planning evaluation
(hybrid re-score stats in planning/<planner>/summary.json).


## Planner Smoke Test

End-to-end check on the synthetic point-mass task, where the global model is exact: every planner must return bounded actions, the local planners' optimization cost must decrease, and hybrid refinement must pass the global re-score. Wall time ~1-2 minutes on CPU.

In [16]:
smoke_root = Path('/content/lg_smoke') if Path('/content').exists() else REPO / 'runs/local_global'
run_if_safe(['bash', str(REPO / 'scripts/local_global/run_smoke.sh')],
            env={'LG_SMOKE_ROOT': str(smoke_root), 'PYTHON': sys.executable})

$ LG_SMOKE_ROOT=/content/lg_smoke PYTHON=/usr/bin/python3 bash /content/World_Models_LAS/scripts/local_global/run_smoke.sh
== local/global smoke: configs/local_global/smoke_synthetic.yaml -> /content/lg_smoke/smoke_synthetic ==
Task:          synthetic
Latent cache:  /content/lg_smoke/_synthetic/latent_cache
Action data:   /content/lg_smoke/_synthetic/action_data
Cache manifest present: False (/content/lg_smoke/_synthetic/latent_cache/wm_poc_latent_manifest.json)
Dry run: no files written.
Task:          synthetic
Latent cache:  /content/lg_smoke/_synthetic/latent_cache
Action data:   /content/lg_smoke/_synthetic/action_data
Generated synthetic latent task under /content/lg_smoke/_synthetic
Exported manifest: /content/lg_smoke/smoke_synthetic/transition_data/manifest.json (616 train / 56 val windows, 12 episodes)
Config: configs/local_global/smoke_synthetic.yaml | task=synthetic | max_steps=200
Run dir: /content/lg_smoke/smoke_synthetic | device=cpu | params=8,680 trainable | 392 train

<Popen: returncode: 0 args: ['bash', '/content/World_Models_LAS/scripts/loca...>

## Planning Evaluation

Compares the configured planners (`global_cem`, `local_gd`/`local_adam`, `local_cem`, hybrid with and without global re-score rejection) on offline latent goal-reaching episodes sampled from held-out validation episodes. The global DINO-WM model is both the MPC simulator and the scorer - this structurally favors `global_cem`, so read local/hybrid results primarily through wall time, backward-step counts, and the re-score acceptance rate. The cell is **live and self-gating**: planners with an existing `summary.json` are skipped.

In [17]:
import json

planners = list(lg_config['evaluation']['planners'])
target_episodes = int(lg_config['evaluation']['num_episodes'])


def _planner_done(planner):
    # Done only if its summary.json was produced at >= the configured episode
    # count, so raising num_episodes (e.g. 50 -> 200) re-runs under-sampled
    # planners; per-episode resume continues them from what is already logged.
    sp = RUN_DIR / 'planning' / planner / 'summary.json'
    if not sp.is_file():
        return False
    try:
        s = json.loads(sp.read_text())
    except json.JSONDecodeError:
        return False
    return int(s.get('episodes_requested', 0)) >= target_episodes


missing = [p for p in planners if not _planner_done(p)]
done = [p for p in planners if p not in missing]
for planner in done:
    summary = json.loads((RUN_DIR / 'planning' / planner / 'summary.json').read_text())
    print(f"done: {planner}: success_rate={summary.get('success_rate')} "
          f"norm_dist={summary.get('mean_normalized_final_distance')} "
          f"(n={summary.get('episodes_requested')})")

needs_local = any(p != 'global_cem' for p in missing)
local_ready = (RUN_DIR / 'checkpoints' / 'local_best.pt').is_file()
global_run_output = Path(os.environ['DINO_CKPT_ROOT']) / 'outputs' / str(global_cfg['model_name'])
global_ready = global_cfg['source'] == 'synthetic' or (
    (global_run_output / 'hydra.yaml').is_file() and bool(loadable_checkpoints(global_run_output))
)
if not missing:
    print(f'All planners evaluated at n>={target_episodes}; delete a planning/<planner> dir to redo one.')
elif not global_ready:
    print('Global DINO-WM checkpoint not usable (needs hydra.yaml AND a loadable model_*.pth);')
    print('see the dependency-verification cell above for usable runs / LG_GLOBAL_MODEL_NAME.')
    print('Skipping the launch.')
elif needs_local and not local_ready:
    print('Local surrogate checkpoint missing; run the training cell above first. Skipping the launch.')
else:
    if global_cfg['source'] == 'dino_wm':
        # Idempotent: upstream deps (Colab) + the latent-bypass patch the
        # in-process rollout needs.
        if Path('/content').exists():
            run_if_safe([sys.executable, str(REPO / 'scripts/dino_wm/install_colab_deps.py'), '--quiet'])
        run_if_safe([sys.executable, str(REPO / 'scripts/dino_wm/patch_latent_cache.py')])
    eval_cmd = [sys.executable, str(REPO / 'scripts/local_global/run_planning_eval.py'),
                '--config', CONFIG, '--run-dir', RUN_DIR, '--planners', *missing]
    run_if_safe([*eval_cmd, '--dry-run'])
    enable_local_global_runs()
    run_lg(eval_cmd)

done: global_cem: success_rate=0.97 norm_dist=0.13840896536729036 (n=100)
done: local_cem: success_rate=0.45 norm_dist=0.6745333815966926 (n=100)
done: local_gd: success_rate=0.13 norm_dist=1.3056728870786092 (n=100)
done: local_adam: success_rate=0.41 norm_dist=0.6986892023737599 (n=100)
done: hybrid_cem_local_refine: success_rate=0.54 norm_dist=0.6319047128389382 (n=100)
done: hybrid_cem_local_refine_global_rescore: success_rate=0.97 norm_dist=0.1381835395995974 (n=100)
All planners evaluated at n>=100; delete a planning/<planner> dir to redo one.


## Summary Table and Artifact Links

In [18]:
run_if_safe([sys.executable, str(REPO / 'scripts/local_global/summarize_runs.py'), '--run-root', LG_RUN_ROOT])
summary_csv = LG_RUN_ROOT / '_summary' / 'summary.csv'
if summary_csv.is_file():
    try:
        import pandas as pd
        display(pd.read_csv(summary_csv))
    except Exception:
        print(summary_csv.read_text())
else:
    print('No summary CSV yet; run training/planning first.')
print(f"Artifacts root: {LG_RUN_ROOT}")
for sub in ('transition_data', 'checkpoints', 'metrics', 'planning'):
    target = RUN_DIR / sub
    print(f"  {target} (exists: {target.is_dir()})")

$ /usr/bin/python3 /content/World_Models_LAS/scripts/local_global/summarize_runs.py --run-root /content/drive/MyDrive/wm_poc/logs/local_global
Summarized 7 rows from /content/drive/MyDrive/wm_poc/logs/local_global -> /content/drive/MyDrive/wm_poc/logs/local_global/_summary/summary.csv
Planners seen: global_cem, hybrid_cem_local_refine, hybrid_cem_local_refine_global_rescore, local_adam, local_cem, local_gd
[exit code 0]


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Next Commands

In [ ]:
print(f"""Copy-paste commands (run from {REPO}).

Both configs run the SAME full experiment into the SAME run dir
({LG_RUN_ROOT}/pointmaze_local_full_seed0); pick the file matching your GPU
(throughput knobs only - a T4 runs the identical experiment, just slower,
and the wall-clock caps + planner-level resume handle session limits):

# Surrogate training (resumable across sessions/GPU tiers)
RUN_LOCAL_GLOBAL=1 python scripts/local_global/train_local_surrogate.py \\
  --config configs/local_global/pointmaze_surrogate_a100.yaml \\
  --run-dir {LG_RUN_ROOT}/pointmaze_local_full_seed0
# (on a T4: swap in configs/local_global/pointmaze_surrogate_t4.yaml)

# Planner comparison - re-run the same command to continue after a wall-clock cap
RUN_LOCAL_GLOBAL=1 python scripts/local_global/run_planning_eval.py \\
  --config configs/local_global/pointmaze_surrogate_a100.yaml \\
  --run-dir {LG_RUN_ROOT}/pointmaze_local_full_seed0

# Aggregate everything
python scripts/local_global/summarize_runs.py --run-root {LG_RUN_ROOT}

# PushT is deliberately deferred until PointMaze passes a real-latent run
# (the latent cache pipeline currently supports point_maze only).""")